In [ ]:
# === STEP 1: MOUNT AND VERIFY SOURCE ===
from google.colab import drive
import os
from glob import glob

drive.mount('/content/drive')

# Exact paths from your previous successful verification
source_base = '/content/drive/MyDrive/MHCNNFD_Data/'
source_folders = [
    'Evening_Fire_Incident_aug_img',
    'Evening_Forest_Condition_aug_img',
    'Pre-_Evening_Fire_Incident_aug_img',
    'Pre-_Evening_Forest_Condition_aug_img'
]

print("🔍 Verifying Original Dataset...")
print("-" * 50)

total_found = 0
folder_status = True

for folder in source_folders:
    path = os.path.join(source_base, folder)
    if os.path.exists(path):
        # Counting only PNGs as per your notebook's last state
        count = len(glob(os.path.join(path, '*.png')))
        total_found += count
        print(f"✅ {folder:<40} | Count: {count}")
    else:
        print(f"❌ {folder:<40} | NOT FOUND")
        folder_status = False

print("-" * 50)
print(f"TOTAL IMAGES FOUND: {total_found}")
print(f"EXPECTED: 15,560")

if total_found == 15560 and folder_status:
    print("\n🟢 VERIFICATION SUCCESSFUL. Ready for Step 2 (33% Sampling).")
else:
    print("\n🔴 VERIFICATION FAILED. Please check the paths or folder names.")

Mounted at /content/drive
🔍 Verifying Original Dataset...
--------------------------------------------------
✅ Evening_Fire_Incident_aug_img            | Count: 3890
✅ Evening_Forest_Condition_aug_img         | Count: 3890
✅ Pre-_Evening_Fire_Incident_aug_img       | Count: 3890
✅ Pre-_Evening_Forest_Condition_aug_img    | Count: 3890
--------------------------------------------------
TOTAL IMAGES FOUND: 15560
EXPECTED: 15,560

🟢 VERIFICATION SUCCESSFUL. Ready for Step 2 (33% Sampling).


In [ ]:
# === STEP 2: STRATIFIED 33.33% SAMPLING ===
import pandas as pd
from sklearn.model_selection import train_test_split

# Create a list of all image paths and their corresponding labels
all_image_paths = []
all_labels = []

for folder in source_folders:
    class_dir = os.path.join(source_base, folder)
    paths = glob(os.path.join(class_dir, '*.png'))
    all_image_paths.extend(paths)
    all_labels.extend([folder] * len(paths))

# Create a master DataFrame
df_master = pd.DataFrame({'path': all_image_paths, 'label': all_labels})

# Use train_test_split to grab exactly 33.33% of the data
# We use 'stratify' to ensure an equal amount is taken from each of the 4 folders
_, df_subset = train_test_split(
    df_master,
    test_size=0.3333,
    random_state=42,
    stratify=df_master['label']
)

print(f"🎯 Sampling Complete!")
print(f"Total images selected: {len(df_subset)}")
print("\nBreakdown per class:")
print(df_subset['label'].value_counts())

# Verification logic
expected_total = int(15560 * 0.3333)
if abs(len(df_subset) - expected_total) < 5:
    print(f"\n🟢 SUCCESS: Sampled {len(df_subset)} images (~33.33%).")
    print("Ready for Step 3: The 5-way Augmentation.")
else:
    print(f"\n🔴 WARNING: Sample count ({len(df_subset)}) is off.")

🎯 Sampling Complete!
Total images selected: 5187

Breakdown per class:
label
Pre-_Evening_Forest_Condition_aug_img    1297
Pre-_Evening_Fire_Incident_aug_img       1297
Evening_Fire_Incident_aug_img            1297
Evening_Forest_Condition_aug_img         1296
Name: count, dtype: int64

🟢 SUCCESS: Sampled 5187 images (~33.33%).
Ready for Step 3: The 5-way Augmentation.


In [ ]:
# === STEP 3: PATH-BASED SPLITTING ===
from sklearn.model_selection import train_test_split

# Split our 5,187 unique images
train_df, temp_df = train_test_split(df_subset, test_size=0.2, random_state=42, stratify=df_subset['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"✅ Split logic defined:")
print(f"  - Train: {len(train_df)} unique images (will become {len(train_df)*5})")
print(f"  - Val:   {len(val_df)} unique images (will become {len(val_df)*5})")
print(f"  - Test:  {len(test_df)} unique images (will become {len(test_df)*5})")

✅ Split logic defined:
  - Train: 4149 unique images (will become 20745)
  - Val:   519 unique images (will become 2595)
  - Test:  519 unique images (will become 2595)


In [ ]:
import cv2
import numpy as np
import os
from tqdm.auto import tqdm # Progress bar library

# Define the new root folder
robust_base = '/content/drive/MyDrive/Robust_MHCNNFD_Dataset'

def generate_robust_set(df, split_name):
    print(f"\n📂 Starting {split_name} set generation...")

    # tqdm creates the real-time progress bar
    # total=len(df) means it counts the 5,187 unique images
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        img_path = row['path']
        label = row['label']
        fname = os.path.basename(img_path)

        # Create destination folder
        dest_dir = os.path.join(robust_base, split_name, label)
        os.makedirs(dest_dir, exist_ok=True)

        # Load the original image
        img = cv2.imread(img_path)
        if img is None:
            continue

        # Generate and Save all 5 versions
        # 1. Original
        cv2.imwrite(os.path.join(dest_dir, f"orig_{fname}"), img)

        # 2. Light Gaussian (std=10)
        noise_l = np.random.normal(0, 10, img.shape).astype(np.uint8)
        cv2.imwrite(os.path.join(dest_dir, f"ltg_{fname}"), cv2.add(img, noise_l))

        # 3. Heavy Gaussian (std=25)
        noise_h = np.random.normal(0, 25, img.shape).astype(np.uint8)
        cv2.imwrite(os.path.join(dest_dir, f"hvg_{fname}"), cv2.add(img, noise_h))

        # 4. Motion Blur (9x9)
        cv2.imwrite(os.path.join(dest_dir, f"blr_{fname}"), cv2.GaussianBlur(img, (9, 9), 0))

        # 5. Fog (70% overlay)
        overlay = np.full(img.shape, 180, dtype=np.uint8)
        cv2.imwrite(os.path.join(dest_dir, f"fog_{fname}"), cv2.addWeighted(img, 0.3, overlay, 0.7, 0))

# Run the generation with real-time bars
generate_robust_set(train_df, 'train')
generate_robust_set(val_df, 'val')
generate_robust_set(test_df, 'test')

print("\n✨ SUCCESS! Your Robust_Training folder is fully populated.")
print(f"Final Location: {robust_base}")

NameError: name 'train_df' is not defined

In [ ]:
import os
import pandas as pd

# Path to your new dataset
robust_base = '/content/drive/MyDrive/Robust_MHCNNFD_Dataset'
splits = ['train', 'val', 'test']
classes = [
    'Evening_Fire_Incident_aug_img',
    'Evening_Forest_Condition_aug_img',
    'Pre-_Evening_Fire_Incident_aug_img',
    'Pre-_Evening_Forest_Condition_aug_img'
]

print("📊 FINAL DATASET AUDIT: Robust_MHCNNFD_Dataset")
print("="*70)

audit_results = []

for split in splits:
    split_total = 0
    print(f"\n📂 SPLIT: {split.upper()}")

    for cls in classes:
        path = os.path.join(robust_base, split, cls)
        if os.path.exists(path):
            files = os.listdir(path)
            count = len(files)
            split_total += count

            # Count specific stress types using the prefixes we defined
            orig = len([f for f in files if f.startswith('orig_')])
            ltg = len([f for f in files if f.startswith('ltg_')])
            hvg = len([f for f in files if f.startswith('hvg_')])
            blr = len([f for f in files if f.startswith('blr_')])
            fog = len([f for f in files if f.startswith('fog_')])

            print(f"  {cls[:30]:<30} | Total: {count:5d} | (O:{orig}, L:{ltg}, H:{hvg}, B:{blr}, F:{fog})")

            audit_results.append({
                'Split': split, 'Class': cls, 'Total': count,
                'Original': orig, 'Light_G': ltg, 'Heavy_G': hvg, 'Blur': blr, 'Fog': fog
            })
        else:
            print(f"  ❌ {cls[:30]:<30} | FOLDER MISSING")

    print(f"  {'':<30} | SUB-TOTAL: {split_total}")

print("\n" + "="*70)

# Final Validation
df_audit = pd.DataFrame(audit_results)
grand_total = df_audit['Total'].sum()

print(f"✅ GRAND TOTAL IMAGES: {grand_total:,}")
print(f"🎯 EXPECTED TOTAL:    25,935")

if grand_total == 25935:
    print("\n🟢 VERIFICATION PASSED: The dataset is 100% complete and perfectly balanced.")
else:
    diff = 25935 - grand_total
    print(f"\n⚠️ WARNING: Missing {diff} images. This is usually due to Google Drive sync delays.")

📊 FINAL DATASET AUDIT: Robust_MHCNNFD_Dataset

📂 SPLIT: TRAIN
  Evening_Fire_Incident_aug_img  | Total:  5190 | (O:1038, L:1038, H:1038, B:1038, F:1038)
  Evening_Forest_Condition_aug_i | Total:  5185 | (O:1037, L:1037, H:1037, B:1037, F:1037)
  Pre-_Evening_Fire_Incident_aug | Total:  5185 | (O:1037, L:1037, H:1037, B:1037, F:1037)
  Pre-_Evening_Forest_Condition_ | Total:  5185 | (O:1037, L:1037, H:1037, B:1037, F:1037)
                                 | SUB-TOTAL: 20745

📂 SPLIT: VAL
  Evening_Fire_Incident_aug_img  | Total:   645 | (O:129, L:129, H:129, B:129, F:129)
  Evening_Forest_Condition_aug_i | Total:   650 | (O:130, L:130, H:130, B:130, F:130)
  Pre-_Evening_Fire_Incident_aug | Total:   650 | (O:130, L:130, H:130, B:130, F:130)
  Pre-_Evening_Forest_Condition_ | Total:   650 | (O:130, L:130, H:130, B:130, F:130)
                                 | SUB-TOTAL: 2595

📂 SPLIT: TEST
  Evening_Fire_Incident_aug_img  | Total:   650 | (O:130, L:130, H:130, B:130, F:130)
  Evening_Fo

NameError: name 'model' is not defined

In [ ]:
# Cell 1: Mount Drive and Imports
from google.colab import drive
import tensorflow as tf
import os

drive.mount('/content/drive')
print("✅ Drive mounted and ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted and ready.
